# 07 · 读数预览容错与 MC 采样耗时

本 notebook 对应 **C-39** 对话结论（见 `PROGRESS.md` / `OBSERVATIONS.md` Checkpoint #35）：

1. **候选预览** 与 **MC 后验** 是两条路径 — 均价/均格主要只影响枚举预览与分析估算。
2. OCR 残留 `gold_cells=0` + `gold_avg_value` 会让预览误报；`relax_bucket_for_enumeration_preview` 仅用于 UI。
3. 推断慢时先看 `sample_ms`（`sample_session_truth` × N），不是 `filter_ms`。

在仓库根目录执行：`jupyter notebook notebooks/07_capture_readings_and_mc_perf.ipynb`

## 1. 环境

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from bidking_lab.extract.bid_map_table import load_bid_map_table
from bidking_lab.extract.drop_table import load_drop_table
from bidking_lab.extract.item_table import load_item_table
from bidking_lab.inference.ground_truth import sample_session_truth
from bidking_lab.inference.observation import (
    QualityBucketObs,
    candidates_for_bucket,
    relax_bucket_for_enumeration_preview,
)

TABLES = REPO / "data" / "raw" / "tables"
maps = load_bid_map_table(TABLES / "BidMap.txt")
drops = load_drop_table(TABLES / "Drop.txt")
items = load_item_table(TABLES / "Item.txt")
print(f"loaded {len(maps)} maps, {len(drops)} pools, {len(items)} items")

## 2. 金品：手填总格 vs OCR 残留（仅均价）

仓库容量 120；其它 bucket 已占 0 格（简化）。

In [ ]:
WH = 120

hand_fill = QualityBucketObs(quality=5, total_cells=26)
c_hand = candidates_for_bucket(hand_fill, warehouse_capacity=WH)
print(f"手填 gold_cells=26: {len(c_hand)} 候选, top-3 = {[(c.total_cells, c.count) for c in c_hand[:3]]}")

ocr_dirty = QualityBucketObs(quality=5, total_cells=0, avg_value=35100)
c_strict = candidates_for_bucket(ocr_dirty, warehouse_capacity=WH)
print(f"严格 OCR 残留 (cells=0, avg=35100): {len(c_strict)} 候选")

relaxed, dropped = relax_bucket_for_enumeration_preview(ocr_dirty, warehouse_capacity=WH)
c_relax = candidates_for_bucket(relaxed, warehouse_capacity=WH)
print(f"预览 relax 丢弃 {dropped}: {len(c_relax)} 候选, top-3 = {[(c.total_cells, c.count) for c in c_relax[:3]]}")

## 3. 按地图计时 MC 采样（冷启动体感）

与 Streamlit `_sample_truths_cached` 相同核心：`sample_session_truth` × `n_trials`。
第一次换到某 `map_id` 往往最慢（无 Streamlit 缓存时）。

In [ ]:
N_TRIALS = 1000
SEED = 42
PROBE_MAPS = [2402, 2401, 2501, 2403]

rng = np.random.default_rng(SEED)
for mid in PROBE_MAPS:
    t0 = time.perf_counter()
    for _ in range(N_TRIALS):
        sample_session_truth(mid, maps=maps, drops=drops, items=items, rng=rng)
    ms = (time.perf_counter() - t0) * 1000
    name = maps[mid].name if mid in maps else str(mid)
    print(f"map {mid} ({name[:12]}…): {ms/1000:.2f}s for {N_TRIALS} trials  (~{ms/N_TRIALS:.1f} ms/trial)")

## 4. 小结

| 现象 | 对应 |
|------|------|
| 预览 ⚠️ 但 MC 仍跑 | 预览 `relax` / 脏 `obs`；MC 不看 `avg_value` |
| 同图第二次推断更快 | `@st.cache_data` 命中 `_sample_truths_cached` |
| 日志里 `filter_ms`≈0、`sample_ms` 很大 | 正常；见 TROUBLESHOOTING #41 |

实机调试：`$env:BIDKING_AGENT_DEBUG=1` 后查看 `debug-*.log` 中 `"MC timing"` 行。